In [ ]:
import subprocess, sys, os, glob, shutil, time
SCRIPT = '# -*- coding: utf-8 -*-\n"""So ghép cặp trực tiếp phần đổi phần tử giữa các lượt lưới RL (report/320 mục 14.5) — 0 GPU. Hậu kiểm, chọn sau khi thấy số.\n\nMỗi bước, so với S1/101: phần tử = +1 nếu cứu mà S1 đúng loại, −1 nếu phá mà nhánh đúng loại; loại thao tác = phần cứu/phá còn lại.\nHai cách đọc độc lập nhau:\n  qua_s1    hiệu điểm từng bước giữa hai nhánh (có S1 làm mốc); KTC bootstrap cụm app, p dấu chính xác trên bước khác nhau.\n  truc_tiep exec của hai nhánh trên bước cả hai đúng loại thao tác (không qua S1); KTC bootstrap, McNemar chính xác.\nHolm tính trong từng họ. Tự kiểm: tổng từng bước phải ra đúng phan_ra.json (ck500 +28/+41). Ghi runs/luoi/test_2_11/phan_tu_ghep_cap.json.\n"""\nimport json, os, sys\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nfrom scipy.stats import binomtest\nimport score_run as S\n\nK = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "runs")\nT = "luoi/test_2_11/score_%s_101_test_raw.jsonl"\nN = {"ck500": "grpo_spice/score_ck500_test_raw.jsonl", "SCST + SPICE": T % "scst_spice", "GRPO + CIDEr-D": T % "grpo_cider",\n     "GRPO + METEOR": T % "grpo_meteor", "GRPO + ROUGE-L": T % "grpo_rougel"}\nKHAC = ["GRPO + CIDEr-D", "GRPO + METEOR", "GRPO + ROUGE-L"]\nB, SEED = int(os.environ.get("B", 10000)), S.SEED\nnap = lambda f: {(x["episode_id"], x["step_id"]): x for x in map(json.loads, open(os.path.join(K, f), encoding="utf-8"))}\ne = lambda r: int(bool(r.get("executable", 0)))\na = lambda r: int(bool(r.get("action_ok", 0)))\n\n\ndef holm(ps):\n    thu, kq, m = sorted(ps, key=ps.get), {}, 0.0\n    for i, t in enumerate(thu):\n        m = max(m, min(1.0, ps[t] * (len(thu) - i)))\n        kq[t] = m\n    return kq\n\n\ndef main():\n    s1 = nap("score_s1_seed101_raw.jsonl")\n    X = {t: nap(f) for t, f in N.items()}\n    ks = sorted(s1)\n    cum = {k: s1[k].get("app") or f"ep{k[0]}" for k in ks}\n    for t in X:\n        assert set(X[t]) == set(s1), t\n    pt = {t: {k: (e(X[t][k]) - e(s1[k])) * (a(s1[k]) if e(X[t][k]) else a(X[t][k])) for k in ks} for t in X}\n    lt = {t: {k: (e(X[t][k]) - e(s1[k])) - pt[t][k] for k in ks} for t in X}\n    tong = {t: (sum(lt[t].values()), sum(pt[t].values())) for t in X}\n    print("tổng (loại, phần tử):", tong, flush=True)\n    if tong["ck500"] != (28, 41):\n        sys.exit(f"⛔ DỪNG, không tái lập ck500 +28/+41: {tong[\'ck500\']}")\n    if os.path.exists(os.path.join(K, "luoi/test_2_11/phan_ra.json")):\n        pr = json.load(open(os.path.join(K, "luoi/test_2_11/phan_ra.json"), encoding="utf-8"))\n        for t in X:\n            assert tong[t] == (pr[t]["rong_loai"], pr[t]["rong_phan_tu"]), (t, tong[t])\n    print("✅ tái lập phân rã", flush=True)\n\n    def bs(d, tap=None):\n        q = tap if tap is not None else ks\n        p, (lo, hi), _, _ = S.cluster_bootstrap([(k, d[k]) for k in q], lambda u: cum[u[0]], lambda u: u[1], B=B, seed=SEED)\n        hon, kem = sum(d[k] > 0 for k in q), sum(d[k] < 0 for k in q)\n        return dict(buoc_rong=round(p * len(q), 1), diem=round(100 * p, 2), lo=round(100 * lo, 2), hi=round(100 * hi, 2), hon=hon, kem=kem,\n                    n=len(q), p=float(binomtest(hon, hon + kem).pvalue) if hon + kem else 1.0)\n\n    out = dict(B=B, seed=SEED, tong=tong, qua_s1={}, truc_tiep={}, tuong_phan={})\n    for ten, D in (("phan_tu", pt), ("loai", lt)):\n        o = {f"ck500 − {t}": bs({k: D["ck500"][k] - D[t][k] for k in ks}) for t in N if t != "ck500"}\n        for t, h in holm({t: v["p"] for t, v in o.items()}).items():\n            o[t]["p_holm"] = h\n        out["qua_s1"][ten] = o\n        tb = {k: sum(D[t][k] for t in KHAC) / 3 for k in ks}\n        out["tuong_phan"][ten] = {"ck500 − TB ba độ đo khác (cùng GRPO)": bs({k: D["ck500"][k] - tb[k] for k in ks}),\n                                  "TB hai lượt SPICE − TB ba độ đo khác": bs({k: (D["ck500"][k] + D["SCST + SPICE"][k]) / 2 - tb[k] for k in ks})}\n        print(ten, json.dumps(out["qua_s1"][ten], ensure_ascii=False), json.dumps(out["tuong_phan"][ten], ensure_ascii=False), flush=True)\n    o = {}\n    for t in N:\n        if t == "ck500":\n            continue\n        tap = [k for k in ks if a(X["ck500"][k]) and a(X[t][k])]\n        o[f"ck500 − {t}"] = bs({k: e(X["ck500"][k]) - e(X[t][k]) for k in tap}, tap)\n    for t, h in holm({t: v["p"] for t, v in o.items()}).items():\n        o[t]["p_holm"] = h\n    out["truc_tiep"] = o\n    print("trực tiếp", json.dumps(o, ensure_ascii=False), flush=True)\n    json.dump(out, open(os.path.join(K, "luoi/test_2_11/phan_tu_ghep_cap.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)\n    print("✅ xong", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
W = "/kaggle/working/w"; shutil.rmtree(W, ignore_errors=True); os.makedirs(W)
src = [p for p in glob.glob("/kaggle/input/**/luoi-cpu-3-11", recursive=True) if os.path.isdir(p)]; print(src, flush=True); S = src[0]
for d in ("harness", "runs"):
    shutil.copytree(f"{S}/{d}", f"{W}/{d}", dirs_exist_ok=True)
open(f"{W}/harness/phan_tu_ghep_cap.py", "w", encoding="utf-8").write(SCRIPT)
subprocess.run(f"{sys.executable} -m pip install -q scipy 2>&1 | tail -1", shell=True)
log = "/kaggle/working/phan_tu_ghep_cap.log"; t0 = time.time()
p = subprocess.Popen(f"cd {W}/harness && {sys.executable} -u phan_tu_ghep_cap.py > {log} 2>&1", shell=True)
while p.poll() is None:
    time.sleep(60); print(time.strftime("%H:%M"), f"{time.time()-t0:.0f} s", open(log, errors="ignore").read()[-200:].replace("\n", " | "), flush=True)
print("ma thoat", p.returncode, flush=True); print(open(log, errors="ignore").read()[-6000:], flush=True)
f = f"{W}/runs/luoi/test_2_11/phan_tu_ghep_cap.json"
if os.path.exists(f): shutil.copyfile(f, "/kaggle/working/phan_tu_ghep_cap.json")
shutil.rmtree(W); print(os.listdir("/kaggle/working"), flush=True)
assert p.returncode == 0
